Sarah Sullivan

October 23, 2025

PSID 

In [ ]:
# Import necessary packages
import pandas as pd
import numpy as np

In [2]:
# Define root
root = "/Users/sarsul/Library/CloudStorage/Dropbox-UniversityofMichigan/Sarah Sullivan/SARAH-SOFT/research/psid/"

In [ ]:
df = pd.read_stata(root + "01_try_pyth_v8.dta")

In [4]:
df

,ID,yr,hhr_,ages,fam,fam_number,birth_year,bio_mom_id,bio_dad_id,a_mom_id,...,in_2007,in_2009,in_2011,in_2013,in_2015,in_2017,in_2019,in_2021,in_2023,Age
0,1030.0,1973,"1002,1003,1004,1030","51,25,23,1",1.0,30.0,1972.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
1,1030.0,1974,"1002,1003,1004,1030","52,27,25,1",1.0,30.0,1972.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0
2,1030.0,1975,"1002,1003,1004,1030","53,28,26,3",1.0,30.0,1972.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0
3,1030.0,1976,"1002,1003,1004,1030","55,30,28,4",1.0,30.0,1972.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
4,1030.0,1977,"1002,1003,1004,1030","55,30,28,5",1.0,30.0,1972.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
347807,6872185.0,2011,"6872001,6872002,6872003,6872030,6872031,687203...","50,28,23,7,5,5,5,29,27,25,8,5",6872.0,185.0,2006.0,6872184.0,6872174.0,NaN,...,0.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,5.0
347808,6872185.0,2013,"6872001,6872002,6872003,6872030,6872031,687203...","52,30,25,9,7,7,7,31,27,10,7",6872.0,185.0,2006.0,6872184.0,6872174.0,NaN,...,0.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,7.0
347809,6872185.0,2015,"6872001,6872002,6872003,6872030,6872031,687203...","54,32,27,11,9,9,9,33,29,12,9",6872.0,185.0,2006.0,6872184.0,6872174.0,NaN,...,0.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,9.0
347810,6872185.0,2017,"6872002,6872003,6872030,6872031,6872033,687203...","56,34,29,13,11,11,11,31,14,11",6872.0,185.0,2006.0,6872184.0,6872174.0,NaN,...,0.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,11.0


In [ ]:
df['hhr_prev'] = df['hhr_'].shift(1)
df['ages_prev'] = df['ages'].shift(1)


In [ ]:
def parse_tuple_string(x):
    if isinstance(x, str):
        try:
            return list(ast.literal_eval(x))
        except (SyntaxError, ValueError):
            return []
    elif isinstance(x, (list, tuple)):
        return list(x)
    else:
        return []

df['hhr_prev'] = df['hhr_prev'].apply(parse_tuple_string)
df['hhr_'] = df['hhr_'].apply(parse_tuple_string)

df['who_left'] = df.apply(lambda row: [x for x in row['hhr_prev'] if x not in row['hhr_']], axis=1)
df['who_came'] = df.apply(lambda row: [x for x in row['hhr_'] if x not in row['hhr_prev']], axis=1)


               ID    yr                                               hhr_  \
0          1030.0  1973                           [1002, 1003, 1004, 1030]   
1          1030.0  1974                           [1002, 1003, 1004, 1030]   
2          1030.0  1975                           [1002, 1003, 1004, 1030]   
3          1030.0  1976                           [1002, 1003, 1004, 1030]   
4          1030.0  1977                           [1002, 1003, 1004, 1030]   
...           ...   ...                                                ...   
347807  6872185.0  2011  [6872001, 6872002, 6872003, 6872030, 6872031, ...   
347808  6872185.0  2013  [6872001, 6872002, 6872003, 6872030, 6872031, ...   
347809  6872185.0  2015  [6872001, 6872002, 6872003, 6872030, 6872031, ...   
347810  6872185.0  2017  [6872002, 6872003, 6872030, 6872031, 6872033, ...   
347811  6872185.0  2019  [6872001, 6872002, 6872003, 6872030, 6872031, ...   

                                 ages     fam  fam_number mom_m

In [34]:
df['who_came'] = df.apply(
    lambda row: [] if not row['hhr_prev'] else row['who_came'],
    axis=1
)

In [36]:

def get_ages_left(row):
    hhr_prev = row['hhr_prev'] if isinstance(row['hhr_prev'], list) else []
    ages_prev = row['ages_prev'] if isinstance(row['ages_prev'], list) else []
    who_left = row['who_left'] if isinstance(row['who_left'], list) else []
    return [ages_prev[hhr_prev.index(pid)] for pid in who_left if pid in hhr_prev and hhr_prev.index(pid) < len(ages_prev)]

def get_ages_came(row):
    hhr_ = row['hhr_'] if isinstance(row['hhr_'], list) else []
    ages = row['ages'] if isinstance(row['ages'], list) else []
    who_came = row['who_came'] if isinstance(row['who_came'], list) else []
    return [ages[hhr_.index(pid)] for pid in who_came if pid in hhr_ and hhr_.index(pid) < len(ages)]

df['ages_left'] = df.apply(get_ages_left, axis=1)
df['ages_came'] = df.apply(get_ages_came, axis=1)


In [39]:
df['adult_came'] = df['ages_came'].apply(
    lambda ages: int(isinstance(ages, list) and any(age >= 18 for age in ages))
)

df['child_came'] = df['ages_came'].apply(
    lambda ages: int(isinstance(ages, list) and any(age < 18 for age in ages))
)

df['adult_left'] = df['ages_left'].apply(
    lambda ages: int(isinstance(ages, list) and any(age >= 18 for age in ages))
)

df['child_left'] = df['ages_left'].apply(
    lambda ages: int(isinstance(ages, list) and any(age < 18 for age in ages))
)

In [40]:
df['n_adults_left'] = df['ages_left'].apply(
    lambda ages: sum(age >= 18 for age in ages) if isinstance(ages, list) else 0
)

df['n_adults_came'] = df['ages_came'].apply(
    lambda ages: sum(age >= 18 for age in ages) if isinstance(ages, list) else 0
)

In [47]:
df.to_csv('final_data_v1.csv', index=False)